In [1]:
import pyspark
print(pyspark.__version__)

4.0.1


In [2]:
from pyspark.sql import SparkSession
import os
# 1. Direct Spark to find winutils.exe inside your existing manual Hadoop 3 environment layout
os.environ["HADOOP_HOME"] = "C:\\hadoop"
os.environ["spark.home.dir"] = "C:\\Spark\\spark-4.0.1-bin-hadoop3"
spark = (SparkSession.builder
        .appName("Practical_Test")
        .master("local[4]")
        .config("spark.executor.memory", "8g")
        .config("spark.driver.memory", "4g")
        .config("spark.python.worker.faulthandler.enabled", "true")
        .config("spark.jars.packages", "io.delta:delta-spark_2.13:4.0.1")
        .config("spark.sql.execution.pyspark.udf.faulthandler.enabled", "true")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.sql.debug.maxToStringFields", 2000).getOrCreate())

In [3]:
from pyspark.sql.types import StructField, StructType, StringType, ArrayType

tickets_schema = StructType([
    StructField("ticket_id", StringType()),
StructField("created_at", StringType()),
    StructField("updated_at", StringType()),
StructField("resolved_at", StringType()),
StructField("status", StringType()),
StructField("priority", StringType()),
StructField("channel", StringType()),
StructField("product", StringType()),
StructField("subject", StringType()),
StructField("description", StringType()),
StructField("requester", StructType([StructField("customer_id", StringType()), StructField("email", StringType()), StructField("locale", StringType())])), # STRUCT<customer_id:STRING,email:STRING,locale:STRING>,
StructField("assignee_id", StringType()),
StructField("tags", ArrayType(StringType())),
StructField("category", StringType()),
StructField("reopen_count", StringType()),
StructField("csat", StringType()),
StructField("schema_version", StringType()),
StructField("_corrupt_record", StringType()),
])

tickets_df = spark.read.format("json").option("encoding", "UTF-8").option("recursiveFileLookup", True).option("pathGlobFilter", "*.jsonl").option("columnNameOfCorruptRecord", "_corrupt_record").schema(tickets_schema).load("D:\\Project_AI\\SupportIQ\\.lake\\landing\\helpdesk\\tickets\\arrival_date=2026-06-18").select("ticket_id", "status", "priority")
tickets_df.count()

10983

1. SOLVED or CLOSED → CLOSED_NO_FOLLOW_UP
2. PENDING or ON-HOLD → WAITING_FOR_CUSTOMER
3. NEW or OPEN with priority URGENT or HIGH → ESCALATE
4. NEW or OPEN with priority NORMAL or LOW → STANDARD_QUEUE
5. All other or missing combinations → NEEDS_REVIEW

In [4]:
from pyspark.sql.functions import col, when, trim, upper

tickets_df = tickets_df.withColumn("priority_standardized", upper(trim(col("priority")))).withColumn("status_standardized", upper(trim(col("status"))))
tickets_df = tickets_df.withColumn("queue_action", when(col("status_standardized").isin("SOLVED","CLOSED"), "CLOSED_NO_FOLLOW_UP")
                                   .when(col("status_standardized").isin("PENDING", "ON-HOLD"), "WAITING_FOR_CUSTOMER")
                                   .when((col("status_standardized").isin("NEW", "OPEN")) & (col("priority_standardized").isin("URGENT", "HIGH")), "ESCALATE")
                                   .when((col("status_standardized").isin("NEW", "OPEN")) & (col("priority_standardized").isin("NORMAL", "LOW")), "STANDARD_QUEUE")
                                   .otherwise("NEEDS_REVIEW")).drop("priority_standardized", "status_standardized")

In [5]:
tickets_df.write.format("delta").mode("overwrite").option("path", "file:///D:/Project_AI/SupportIQ/.lake/interview/focused/FP-001_queue_action").saveAsTable("FP_001_queue_action")

In [6]:
from pyspark.sql.functions import count
tickets_df.groupBy(col("queue_action")).agg(count("*")).show()

+--------------------+--------+
|        queue_action|count(1)|
+--------------------+--------+
|      STANDARD_QUEUE|    7702|
|        NEEDS_REVIEW|     119|
|WAITING_FOR_CUSTOMER|     500|
| CLOSED_NO_FOLLOW_UP|    1223|
|            ESCALATE|    1439|
+--------------------+--------+



In [7]:
spark.table("FP_001_queue_action").count() # Reading from the catalog

10983

In [8]:
validate_tickets_df = spark.read.format("delta").load("file:///D:/Project_AI/SupportIQ/.lake/interview/focused/FP-001_queue_action")

In [9]:
validate_tickets_df.schema

StructType([StructField('ticket_id', StringType(), True), StructField('status', StringType(), True), StructField('priority', StringType(), True), StructField('queue_action', StringType(), True)])

In [10]:
validate_tickets_df.count()

10983

In [11]:
validate_tickets_df.groupBy(col("queue_action")).agg(count("*")).show()

+--------------------+--------+
|        queue_action|count(1)|
+--------------------+--------+
|      STANDARD_QUEUE|    7702|
|        NEEDS_REVIEW|     119|
|WAITING_FOR_CUSTOMER|     500|
| CLOSED_NO_FOLLOW_UP|    1223|
|            ESCALATE|    1439|
+--------------------+--------+

